# Token-Level Watermark Visualization

This notebook reproduces the `cli_watermark.py` workflow with rich visual output.
Each token is colored **green** if it falls in the green-list (watermark key matches)
or left uncolored if it falls in the red-list.

## Color Scheme
- 🟩 **Green** — token is in the green-list (watermark key matched)
- ⬜ **No color** — token is in the red-list (watermark key did not match)

In [1]:
import hashlib
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from colorama import Fore, Style, init as colorama_init

from target_hash_gen.core import g_score, DEFAULT_SEED, _tok, Model, _rng_for, _nucleus, _dist, EOS_ID
from target_hash_gen.greedy import GreedyGenerator
from target_hash_gen.watermark import BoostWatermarkGenerator

# Enable colorama for terminals
colorama_init(autoreset=True)
from IPython.display import display, HTML

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [2]:
def colored_text_html(ids: list[int], tok, seed: str) -> str:
    """Decode tokens with each colored green (green-list) or red (not) — returns HTML."""
    parts: list[str] = []
    for t in ids:
        is_green = g_score(seed, int(t))
        text = tok.decode([t], skip_special_tokens=True)
        # Escape HTML special chars
        text = text.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
        color = 'green' if is_green else '#ff4444'
        parts.append(f'<span style="color:{color};background-color:white;font-size:1.5em;font-weight:{"bold" if is_green else "normal"}">{text}</span>')
    return "".join(parts)


def split_starts(tokens: list[int], tok) -> list[int]:
    """Token indices (1-based) of sentence starts, skipping too-short tails."""
    return [i + 1 for i, t in enumerate(tokens) if tok.decode([t]) in (".", "!", "?") and len(tokens[i + 1 :]) >= 16]

In [3]:
# Parameters
PROMPT = "What is a rainbow?"
MAX_TOKENS = 200
TOP_K = 20
TOP_P = 0.95
SEED = DEFAULT_SEED
WRONG_SEED = "negative key"

display(HTML(f"""
<h3>Configuration</h3>
<table>
<tr><th>Parameter</th><th>Value</th></tr>
<tr><td>Prompt</td><td>{PROMPT}</td></tr>
<tr><td>Seed</td><td>{SEED}</td></tr>
<tr><td>Wrong seed</td><td>{WRONG_SEED}</td></tr>
<tr><td>Max tokens</td><td>{MAX_TOKENS}</td></tr>
<tr><td>Top-k</td><td>{TOP_K}</td></tr>
<tr><td>Top-p</td><td>{TOP_P}</td></tr>
</table>
"""))

Parameter,Value
Prompt,What is a rainbow?
Seed,default-seed
Wrong seed,negative key
Max tokens,200
Top-k,20
Top-p,0.95


In [4]:
# Build prompt
messages = [
    {
        "role": "user",
        "content": PROMPT,
    },
]
prompt = _tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
prompt_ids = _tok(prompt, add_special_tokens=False)["input_ids"]

print(f"Prompt tokens: {len(prompt_ids)}")
print(f"Prompt text: {prompt[:200]}...")

Prompt tokens: 14

Prompt text: <|startoftext|><|im_start|>user
What is a rainbow?<|im_end|>
<|im_start|>assistant
...

In [5]:
# Generate text with all three strategies
gen_wm = BoostWatermarkGenerator(
    seed=SEED,
    top_k=TOP_K,
    top_p=TOP_P,
    delta=1.0,
)
gen_neg = BoostWatermarkGenerator(
    seed=WRONG_SEED,
    top_k=TOP_K,
    top_p=TOP_P,
    delta=1.0,
)
gen_plain = GreedyGenerator(
    top_k=TOP_K,
    top_p=TOP_P,
)

wm = gen_wm.generate(prompt_ids, max_new_tokens=MAX_TOKENS)
neg = gen_neg.generate(prompt_ids, max_new_tokens=MAX_TOKENS)
plain = gen_plain.generate(prompt_ids, max_new_tokens=MAX_TOKENS)
wm, neg, plain = (ids[len(prompt_ids) :] for ids in (wm, neg, plain))

print(f"Generated tokens — watermarked: {len(wm)}, negative-seed: {len(neg)}, plain: {len(plain)}")

Generated tokens — watermarked: 200, negative-seed: 158, plain: 200

In [6]:
# Display watermarked output
display(HTML(f'<h3>🟩 Watermarked output</h3><p><em>Seed: {SEED}</em></p>'))
display(HTML(colored_text_html(wm, _tok, SEED)))

In [7]:
# Display negative-seed output
display(HTML(f'<h3>⬜ Negative-seed output</h3><p><em>Seed: {WRONG_SEED} (evaluated against {SEED})</em></p>'))
display(HTML(colored_text_html(neg, _tok, SEED)))

In [8]:
# Display plain output
display(HTML(f'<h3>⬜ Plain output (baseline)</h3><p><em>No watermark seed applied</em></p>'))
display(HTML(colored_text_html(plain, _tok, SEED)))

In [9]:
# Detection results
display(HTML('<h3>🔍 Detection Results</h3>'))
results = {
    "Watermarked": gen_wm.check_hash(wm, SEED),
    "Negative-seed": gen_wm.check_hash(neg, SEED),
    "Plain": gen_wm.check_hash(plain, SEED),
}
html_rows = "".join(f"<tr><td>{name}</td><td>{val}</td></tr>" for name, val in results.items())
display(HTML(f"""
<table border="1" cellpadding="6">
<tr><th>Text</th><th>Hash Check (seed='{SEED}')</th></tr>
{html_rows}
</table>
"""))

Text,Hash Check (seed='default-seed')
Watermarked,3.6769552621700474
Negative-seed,0.47733437050543776
Plain,-0.14142135623730964


In [10]:
# Hash across splits
display(HTML('<h3>📊 Hash Across Text Splits</h3>'))
for name, text in [("watermarked", wm), ("negative-seed", neg), ("plain", plain)]:
    display(HTML(f'<h4>Splits of {name} text</h4>'))
    html_rows = ""
    for st in split_starts(text, _tok):
        span = text[st:]
        val = gen_wm.check_hash(span, SEED)
        html_rows += f"<tr><td>{st}</td><td>{val}</td></tr>"
    display(HTML(f"""
<table border="1" cellpadding="6">
<tr><th>From token</th><th>Hash (seed='{SEED}')</th></tr>
{html_rows}
</table>
"""))

From token,Hash (seed='default-seed')
25,4.00642341389781
55,3.737046593418298
99,4.278659917902953
137,3.401680257083045
172,1.1338934190276813


From token,Hash (seed='default-seed')
19,0.4240944648399855
50,0.9622504486493759
83,-0.11547005383792493
110,-0.5773502691896261
140,0.9428090415820638


From token,Hash (seed='default-seed')
18,-0.1482498633322197
45,0.40160966445124907
80,-0.36514837167011066
113,-0.32163376045133807
139,-0.1280368799328961
158,0.0


In [11]:
# Summary statistics
display(HTML('<h3>📈 Green-List Token Ratios</h3>'))
stats = []
for name, text in [("watermarked", wm), ("negative-seed", neg), ("plain", plain)]:
    green_count = sum(1 for t in text if g_score(SEED, t))
    ratio = green_count / len(text) if text else 0
    stats.append((name, green_count, len(text), ratio))

html_rows = ""
for name, gc, total, ratio in stats:
    html_rows += f"<tr><td>{name}</td><td>{gc}/{total}</td><td>{ratio:.3f}</td></tr>"
display(HTML(f"""
<table border="1" cellpadding="6">
<tr><th>Text</th><th>Green tokens</th><th>Ratio</th></tr>
{html_rows}
</table>
<p><em>Expected ~0.500 for watermarked, ~0.500 for others</em></p>
"""))

Text,Green tokens,Ratio
watermarked,126/200,0.630
negative-seed,82/158,0.519
plain,99/200,0.495
